# 3. Attention Mechanism 구현

### 4가지 어텐션 메커니즘
- 간소화된 셀프 어텐션
- 셀프 어텐션
- 코잘 어텐션
- 멀티 헤드 어텐션

## 3.1. 긴 시퀀스 모델링의 문제점

- 한 언어에서 다른 언어로 번역할 때, 소스 언어와 타깃 언어의 문법 구조가 달라서 텍스트를 한 단어씩 번역할 수 없음

### 해결책
- Encoder, Decoder (서브모듈 2개)
    - Encoder가 먼저 전체 텍스트를 읽고 처리
    - Decoder가 번역된 텍스트 생성

### 트랜스포머 이전
- RNN이 언어 번역에서 가장 인기 있는 인코더-디코더 구조였음
- RNN은 순차데이터에 잘 맞음

### RNN
- 인코더: 입력 텍스트를 받아서 순차적으로 처리
    - 각 단계마다 은닉 상태(은닉층의 내부값)를 업데이트
    - 최종 은닉 상태로 입력 시퀀스의 전체 의미 포착
    - <span style="color:skyblue">**인코더가 전체 입력 텍스트를 하나의 은닉 상태로 처리함**</span>
- 디코더: 최종 은닉 상태를 받아 한 번에 한 단어씩 번역된 문장 생성
    - 각 단게마다 은닉 상태를 업데이트
    - 이를 통해 다음 단어 예측에 필요한 문맥을 다음 스텝으로 전달
    - <span style="color:skyblue">**디코더가 은닉 상태를 받아 출력을 생성**</span>
- 은닉 상태: 임베딩 벡터 개념

### 인코더-디코더 RNN 제약사항
- 디코딩 단계에서 RNN이 인코더의 이전 은닉 상태를 참조 못함
- 오직 인코더의 최종 은닉 상태에만 의존 (맥락을 놓치기 쉬움)
- 특히 멀리 떨어진 단어에 의존성이 있는 복잡한 문장의 경우가 그럼
- <span style="color:skyblue">**인코더-디코더 RNN의 단점이 어텐션 메커니즘 개발에 동기 부여가 됨**</span>

## 3.2. 어텐션 메커니즘으로 데이터 의존성 포착

- RNN은 짧은 문장 번역은 잘함
- 입력에 있는 이전 단어를 직접 참조하지 못하기 때문에 긴 텍스트 번역은 잘 못함

### [1] RNN을 위한 바흐다나우 어텐션(Bahdanau attention) 메커니즘 
- 인코더-디코더 RNN을 수정하여 디코딩 단계마다 디코더가 선택적으로 입력 시퀀스의 서로 다른 부분 참조 가능

### [2] 원본 트랜스포머 구조
- 3년 후 RNN 구조가 자연어 처리를 위한 DNN에 필수적이지 않다는 걸 발견함
- 바흐다나우 어텐션 메커니즘에서 영감을 받은 셀프 어텐션 메커니즘 포함

#### 셀프 어텐션
- 시퀀스의 표현을 계산할 때 입력 시퀀스에 있는 각 위치가 동일 시퀀스에 있는 다른 모든 위치와의 관련성을 고려하거나 주의를 기울일 수 있음

## 3.3. self-attention으로 입력의 서로 다른 부분에 attention 걸기

* Self 의미
    - 하나의 input sequence에 있는 서로 다른 위치의 원소 사이에서 attention weight을 계산하는 방식
    - input 자체 안에 있는 여러 부분 사이의 관계, 의존성 평가 - 맥락 포착
* 전통적인 어텐션 메커니즘
    - 2개의 다른 시퀀스에 있는 원소 사이의 관계에 초점을 맞춤
    - seq2seq 모델이 입력 시퀀스와 출력 시퀀스 사이에 주의를 기울임

### 3.3.1. 훈련 가능한 가중치가 없는 간단한 self-attention mechanism
* 셀프 어텐션 
    - 입력 시퀀스의 각 원소에 대해 context vector z를 계산하는 것이 목표임
    - z^(2)는 x^(2)와 다른 모든 입력들 사이의 정보를 담은 임베딩

* context vector z
    - 같은 input sequence에 있는 다른 모든 원소의 정보를 통합하여 해당 sequence에 있는 각 원소의 표현을 풍부하게 만듦
    - 문장에서 다른 단어 사이의 관게와 관련성을 이해하게 함

#### 1. self-attention 구현 첫 번째 단게
    - attention score라고 부르는 중간 값 w 계산 (그림의 tensor 값은 소수점 첫번째까지만)
* attention score는 쿼리 x^(2)와 다른 모든 입력 토큰 사이의 dot product로 결정 (내적)
    - 내적은 유사도 척도로 볼 수 있음

In [ ]:
import torch
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your (x^1)
        [0.55, 0.87, 0.66],  # journey (x^2)
        [0.57, 0.85, 0.64],  # starts (x^3)
        [0.22, 0.58, 0.33],  # with (x^4)
        [0.77, 0.25, 0.10],  # one (x^5)
        [0.05, 0.80, 0.55],  # step (x^6)
    ]
)

In [ ]:
query = inputs[1]  # "journey" (x^2)

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(query, x_i)      
print(attn_scores_2)

#### 2. self-attention 구현 두 번째 단게
    - attention score 정규화 
    - attention 가중치의 합이 1이 되도록 함 (LLM 학습의 안정성)

In [ ]:
# 실제로는 softmax 함수를 사용하여 정규화
# softmax 방법이 훈련 과정에 유용한 gradient 속성을 가지고 있기 때문임
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("어텐션 가중치:", attn_weights_2_tmp)
print("어텐션 가중치의 합:", attn_weights_2_tmp.sum())

In [ ]:
# attention score를 정규화하는 softmax 함수의 기본적인 구현
# 해당 softmax 함수는 수치적으로 불안정할 수 있음 (overflow, underflow 문제)

def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2 = softmax_naive(attn_scores_2)
print("어텐션 가중치:", attn_weights_2)
print("어텐션 가중치의 합:", attn_weights_2.sum())

* Softmax 함수
    - attention weight이 항상 양수가 되도록 보장함
    - 출력이 확률이나 상대적인 중요도로 해석 가능해짐
    - 가중치가 높을수록 중요도가 높음

In [ ]:
# naive softmax 함수는 수치적으로 불안정할 수 있음 (overflow, underflow 문제)
# 실전에서는 성능 최적화가 된 pytoruch의 softmax 함수를 사용하는 게 좋음

attn_weights_2 = torch.nn.functional.softmax(attn_scores_2, dim=0)
print("어텐션 가중치:", attn_weights_2)
print("어텐션 가중치의 합:", attn_weights_2.sum())

#### 3. self-attention 구현 세 번째 단계
    - 임베딩된 입력 토큰 x^(i)와 각 토큰에 해당하는 attention weight를 곱한 후 모두 더해서 context vector z^(i) 계산
    - context vector z^(i)는 모든 입력 벡터의 가중치 합이 됨
    - 각 입력 벡터와 어텐션 가중치를 곱하여 구함

In [ ]:
query = inputs[1]  # "journey" (x^2) (두 번째 입력 토큰이 query로 사용됨)
context_vec_2 = torch.zeros(query.shape)  # context vector 초기화
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i  # 가중합 계산
print("context_vector:", context_vec_2)

### 3.3.2. 모든 입력 토큰에 대해 어텐션 가중치 계산

#### 1단계

* for loop를 추가하여 모든 입력 쌍에 대한 점곱 수행

In [ ]:
# 각 입력 토큰이 서로 얼마나 유사한지 계산하는 어텐션 스코어 행렬
# 여기서는 for loop를 사용하여 각 쌍의 입력 토큰에 대해 dot product 계산
# python의 for loop는 느리기 때문에 대신 행렬 곱셈 사용 권장

attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

In [ ]:
# python의 for loop는 느리기 때문에 대신 행렬 곱셈 사용 

attn_scores = inputs @ inputs.T  # 행렬 곱셈을 사용하여 어텐션 스코어 계산
print(attn_scores)

#### 2단계
* 정규화

In [ ]:
# 어텐션 스코어 행렬의 각 행에 대해 softmax 적용하여 어텐션 가중치 계산
# torch.softmax의 dim은 계산 시 input tensor의 차원을 지정
# dim = -1은 마지막 차원에 대해 softmax를 적용한다는 의미 (여기서는 각 행에 대해 softmax 적용)
# attn_scores가 2차원 텐서라면, 열을 따라서 각 행의 값을 모두 더해 1이 되도록 정규화함

attn_weights = torch.softmax(attn_scores, dim=-1)  # 어텐션 스코어 행렬의 각 행에 대해 softmax 적용
print(attn_weights)

In [ ]:
# 가중치 행렬의 각 행의 합이 1이 되는 것을 확인할 수 있음

row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("두 번째 행의 합:", row_2_sum)
print("어텐션 가중치 행렬의 각 행의 합:", attn_weights.sum(dim=-1))

#### 3단계
* attention weights와 input을 행렬 곱셈하여 모든 context vector z들을 계산
* 여기서는 W_{6x6} @ input_embeddings_{6x3} => context_vecs_{6x3}

In [ ]:
all_context_vecs = attn_weights @ inputs  # 행렬 곱셈을 사용하여 모든 context vector 계산
print("모든 context vector:\n", all_context_vecs)

In [ ]:
# 3.3.1절에서 계산한 문맥 벡터 z^(2)와 일치하는 것을 확인할 수 있음

print("3.3.1절에서 계산한 context vector z^(2):", context_vec_2)

## 3.4. 훈련 가능한 가중치를 가진 self-attention 구현
    - scaled dot-product attention
        : GPT 및 대부분의 다른 LLM에서 사용하는 self-attention mechanism
* Query
    - DB의 검색 쿼리와 비슷함
    - 모델이 초점을 맞추거나 이해할 현재 항목을 나타냄 (문장에 있는 단어나 토큰)
    - 쿼리를 사용하여 입력 시퀀스의 다른 부분을 조사하여 얼마나 주의를 기울여야 하는지 결정
* Key
    - DB에서 인덱싱과 검색에 사용되는 키와 비슷함
    - 어텐션 메커니즘에는 입력 시퀀스에 있는 각 항목(ex: 문장에 있는 각 단어)에 연관된 키가 있음 
    - 쿼리와 매칭시키기 위해 사용
* Value
    - DB에 있는 key-value 쌍과 비슷함
    - 입력 항목의 실제 컨텐츠나 표현
    - 모델이 쿼리에 어떤 키가 가장 관련이 있는지 결정하여 attention score를 계산하면 각 value와의 가중치 합을 수행함

* 앞서 구현한 메커니즘과 다른 점
    - 모델 훈련 과정에서 업데이트되는 가중치 행렬 추가
    - 훈련 가능한 가중치 행려을 통해 attention model이 좋은 context vector를 생성하도록 함

### 3.4.1. 단계별로 attention weight 계산 (하나의 문맥 벡터 z^(2) 계산)
    - 훈련 가능한 가중치 행렬 3개 (W_q, W_k, W_v)를 추가하여 구현
    - 3개의 행렬을 사용해 임베딩된 입력 토큰 x^(i)를 각각 Query, Key, Value 벡터로 투영

#### 1단계. x^(2)의 Q, K, V 및 모든 입력 토큰의 K, V 구하기

In [ ]:
x_2 = inputs[1] # "journey" (x^2) (두 번째 입력 원소)
d_in = inputs.shape[1] # 입력 임베딩 크기, d_in = 3
d_out = 2 # 출력 임베딩 크기, d_out = 2
# GPT 같은 모델에서는 일반적으로 입력 차원과 출력 차원이 동일하지만, 여기서는 예시로 출력 차원을 2로 설정

In [ ]:
torch.manual_seed(123)  # 랜덤 시드 설정 (재현 가능성 확보)

# query, key, value 가중치 행렬 초기화 (랜덤 값으로 설정)
# 여기서는 출력을 간단히 하고자 requires_grad=False로 설정하여 학습이 필요 없는 가중치 행렬로 만듦
# 모델 훈련 시에는 requires_grad=True로 설정하여 가중치가 업데이트되도록 해야 함
W_query = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)  # query 가중치 행렬
W_key = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)    # key 가중치 행렬
W_value = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)  # value 가중치 행렬

In [ ]:
# query, key, value 가중치 행렬은 입력 임베딩을 query, key, value 벡터로 변환하는 역할을 함
# W_query는 입력 임베딩을 query 벡터로 변환하는 가중치 행렬, W_key는 입력 임베딩을 key 벡터로 변환하는 가중치 행렬, W_value는 입력 임베딩을 value 벡터로 변환하는 가중치 행렬임

query_2 = x_2 @ W_query  # query 벡터 계산
key_2 = x_2 @ W_key      # key 벡터 계산
value_2 = x_2 @ W_value  # value 벡터 계산
print("query 벡터:", query_2)
print("key 벡터:", key_2)
print("value 벡터:", value_2)

* 주의점
    - W_q, W_k, W_v와 같은 가중치 파라미터는 attention weight과 다른 개념임
    - attention weight은 문맥 벡터가 다른 입력 부분에 의존하는 정도 (네트워크가 입력의 다른 부분에 초점을 맞추는 정도)
    - 가중치 파라미터는 학습되는 coefficient로 네트워크의 연결을 정의함
    - attention weight은 동적이고 문맥에 의존적인 값임

* q^(2)에 대한 attention weight을 계산하려면 모든 입력 원소에 대한 key와 value vector가 필요함

In [ ]:
# 모든 키와 값 구하기

keys = inputs @ W_key  # 모든 key 벡터 계산
values = inputs @ W_value  # 모든 value 벡터 계산
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

#### 2단계. attention score 계산

In [ ]:
# 먼저 attention score w_2 계산

keys_2 = keys[1]  # "journey"에 해당하는 key 벡터
attn_score_22 = query_2.dot(keys_2)  # query 벡터와 key 벡터의 dot product 계산
print("attention score w_2:", attn_score_22)

In [ ]:
# 행렬 곱셈으로 계산 일반화 (모든 query와 모든 key에 대해 attention score 계산)

attn_scores_2 = query_2 @ keys.T  # 주어진 query 벡터에 대한 모든 key 벡터의 dot product 계산
print("attention scores for query_2:", attn_scores_2)

#### 3단계. 정규화
    - attention score를 softmax로 정규화하여 attention weight 계산

    - 다만 attention score를 key의 임베딩 차원의 제곱근으로 나눔

In [ ]:
d_k = keys.shape[-1]  # key 벡터의 임베딩 차원
print("keys.shape:", keys.shape)
print("key 벡터의 임베딩 차원 d_k:", d_k)
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)  # attention weight 계산
print("attention weights for query_2:", attn_weights_2)

#### 4단계. context vector z^(2) 계산
    - value vector에 대한 가중치 합으로 문맥 벡터 계싼
    - attention weight 가중치가 각각의 값 벡터의 중요도에 대한 가중치로 동작

In [ ]:
context_vec_2 = attn_weights_2 @ values  # context vector 계산
print("context vector for query_2:", context_vec_2)

### 3.4.2. self-attention python class 구현

* 아래에서 구현하고 있는 SelfAttention_v1 클래스는 pytorch의 기본 구성 요소인 nn.module 클래스를 상속함

In [ ]:
class SelfAttention_v1(torch.nn.Module):
    # Query, Key, Value 값을 위한 훈련 가능한 가중치 행렬을 초기화 (이 행렬로 입력차원 d_in을 d_out으로 변환)
    def __init__(self, d_in, d_out): 
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

    # forward 메서드로 forward pass를 수행하는 동안 attn_scores를 계산하고 softmax로 정규화하여 attn_weights를 구한 다음, 
    # attn_weights와 value 벡터의 가중합을 계산하여 context vector를 반환
    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
# SelfAttention_v1 클래스 사용 예시
# 두 번째 행이 이전 절에서 구한 context_vec_2와 같다는 것을 확인할 수 있음
torch.manual_seed(123)  # 랜덤 시드 설정 (재현 가능성 확보)
sa_v1 = SelfAttention_v1(d_in, d_out)
print("6개의 문맥벡터\n:", sa_v1(inputs))

#### SelfAttetion_v1 mechanism

* SelfAttention_v1을 pytorch의 nn.Linear 층을 사용하도록 개선 가능함
    - nn.Linear층은 bias unit을 사용하지 않는 경우 행렬 곱셈과 동일한 연산 수행
    - nn.Linear층은 최적화된 가중치 초기화 방법을 사용할 수 있어 모델 훈련을 안정적이고 효율적으로 만듦
    - 수동으로 nn.Parameter(torch.rand(...))을 사용하는 것보다 장점이 큼

In [ ]:
class SelfAttention_v2(torch.nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

In [ ]:
# SelfAttention_v2 클래스 사용 예시
# SelfAttention_v1와 SelfAttention_v2는 가중치 행렬의 초깃값이 다르기 때문에 출력이 다름
# nn.Linear층이 더 복잡한 가중치 초기화 방법을 사용하기 때문임 (nn.Linear는 가중치 행렬을 transpose된 형태로 저장함)
torch.manual_seed(123)  # 랜덤 시드 설정 (재현 가능성 확보)
sa_v2 = SelfAttention_v2(d_in, d_out)
print("6개의 문맥벡터\n:", sa_v2(inputs))

## 3.5. causal attention (masked attention)으로 미래의 단어 감추기
    - masked attention: 주어진 토큰으로 attention score 계산 시 시퀀스의 이전 입력과 현재 입력만 참조 
        * 현재 위치 이전의 토큰만 참조하여 시퀀스의 다음 단어 예측하도록 함
        * 각 토큰 처리 시 현재 토큰 다음에 오는 미래 토큰은 마스킹함
        * 주대각선 위의 attetion weight을 마스킹하고 마스킹하지 않은 attention 
          weight을 정규화하여 각 행의 합이 1이 되게 만듦   
    - 기본적인 self-attention
        * 한 번에 입력 시퀀스 전체를 모두 사용 (즉 미래 단어도 참조하여 계산)

### 3.5.1. causal attetion mask 적용하기

#### 편의상 이전 절에서 만든 SelfAttention_v2 객체의 쿼리와 키 가중치 행렬 재사용

In [ ]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print("attention scores:\n", attn_scores)
print("attention weights:\n", attn_weights)

#### pytorch의 tril 함수로 주대각선 위의 값이 0인 마스크를 만듦

In [ ]:
context_length = attn_scores.shape[0]  # 입력 시퀀스의 길이 (여기서는 6)
mask_simple = torch.tril(torch.ones(context_length, context_length))  # 하삼각 행렬 생성
print("mask_simple:\n", mask_simple)

#### 이 마스크와 atten_weight를 곱해서 주대각선 위의 값을 0으로 만듦

In [ ]:
masked_simple = attn_weights * mask_simple  # 어텐션 가중치에 마스크 적용
print("masked_simple:\n", masked_simple)    

#### 마지막으로 attn_weight을 합이 1이 되도록 다시 정규화 
    - 각 행의 합으로 원소를 나누기
    - 이렇게 만들어진 어텐션 가중치 행렬은 주대각선 위의 값이 0이고, 각 행의 합이 1임

In [ ]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)  # 각 행의 합 계산
masked_simple_norm = masked_simple / row_sums  # 각 행을 해당 행의 합으로 나누어 정규화
print("masked_simple_norm:\n", masked_simple_norm)

### * 참고: 정보누수
    - 어텐션 가중치를 마스킹하고 다시 정규화할 때 미래 토큰의 값이 소프트맥스 계산에 들어가 있어서 (마스킹하려는) 미래의 정보가 현재 토큰에 영향을 미치는 것처럼 보일 수 있음
    - 하지만 마스킹 이후에 어텐션 가중치를 다시 정규화할 때 (마스킹된 위치가 소프트맥스 값에 기여하지 못하므로) 행의 일부분을 사용해 소프트맥스를 다시 계산하는 셈이 됨
    - 소프트맥스 함수의 분모에 모든 위치가 포함되어 있으나, 마스킹 및 재정규화 이후에는 마스킹된 위치의 효과가 사라짐 (소프트맥스 점수에 의미 있는 기여 x)
* 즉, 마스킹 및 재정규화 후에 어텐션 가중치의 분포는 처음 마스킹되지 않은 위치에서 계산된 것과 같음
* 따라서 미래 토큰에서 어떤 정보 누수도 없음

#### causal attention 구현 더 개선
    - 소프트맥스 함수는 입력을 확률 분포로 변환함
    - 한 행에 음의 무한대 값이 있으면 소프트맥스는 해당 값을 0으로 만듦

#### 이처럼 주대각선 위의 값이 1인 마스크를 만들고 1을 -inf로 바꾸는 식으로 더 효율적인 마스킹 기법 구현 가능

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)  # 마스크된 위치에 -inf로 채우기
print("masked:\n", masked)

#### 이 마스킹된 결과에 소프트맥스 함수 적용

In [ ]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)  # 마스크된 어텐션 스코어에 softmax 적용
print("masked attention weights:\n", attn_weights)

#### 이 attn_weights를 사용해 context_vec = attn_weights @ values와 같이 문맥 벡터 계산 가능

### 3.5.2. dropout(드롭아웃)으로 attn_weights에 추가적으로 마스킹하기
    - LLM 훈련 시 과대적합을 줄이기 위해 causal attention에 추가작업이 필요함

* Dropout
    - 훈련 중에 은닉층 유닛을 랜덤으로 선택하여 해당 유닛의 출력을 무시
    - 과대적합 막는 데 도움됨
    - dropout은 훈련 중에만 사용 (그 이후에는 비활성화)
* Transformer 구조에서 일반적으로 attention mechanism에 드롭아웃이 적용되는 두 곳
    - attn_weights 계산 후 (더 널리 쓰이는 방법이라 이 방식 적용 예정)
    - value_vec에 attn_weights를 적용한 후
    

#### 드롭아웃 비율 0.5 (GPT 훈련 시에는 0.1 or 0.2)

In [ ]:
torch.manual_seed(123)  # 랜덤 시드 설정 (재현 가능성 확보)
dropout = torch.nn.Dropout(p=0.5)  # 드롭아웃 레이어 초기화 (드롭아웃 확률 p=0.5)
example = torch.ones(6,6)  # 예시 텐서 (모든 원소가 1인 6x6 텐서)
print("드롭아웃 적용 전:\n", example)
dropped = dropout(example)  # 드롭아웃 적용
print("드롭아웃 적용 후:\n", dropped) 

#### 삭제된 값을 남은 원소들로 보상하기 위해 행렬에서 남은 원소 값을 2배로 늘림
    - p = 0.1이라면 10배
    - p = 0.2라면 5배

In [ ]:
torch.manual_seed(123)  # 랜덤 시드 설정 (재현 가능성 확보)
print(dropout(attn_weights))

### 3.5.3. Causal Attention Class 구현

#### 배치 입력을 시뮬레이션하기 위한 입력 텍스트 샘플 중복 사용

In [ ]:
batch  = torch.stack((inputs, inputs), dim = 0)  # 입력 시퀀스를 배치 차원으로 쌓기
print("배치된 입력 시퀀스의 shape:", batch.shape)

#### CausalAttention Class

In [ ]:
class CausalAttention(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        
        # SelfAttention_v1 클래스와 달리 드롭아웃 층 추가
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, _ = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 첫 번째 차원인 배치 차원은 그대로 유지하면서 두 번째 차원과 세 번째 차원을 바꿈
        attn_scores = queries @ keys.transpose(1, 2)
        
        # 파이토치에서는 밑줄 문자로 끝나는 메서드는 불필요한 메모리 복사를 피하기 위해
        # in-place 연산을 수행하는 메서드임 (예: masked_fill_()는 masked_fill()과 동일하지만 in-place로 동작)
        attn_scores.masked_fill(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf  ),        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values

        return context_vec

* __init__ 메서드에 self.register_buffer() 호출이 추가됨
    - LLM에서 CausalAttention 클래스를 사용할 때 register_buffer() 메서드에 지정한 텐서를 모델과 함께 적절한 장치(CPU or GPU)로 자동으로 이동시킴

#### CausalAttetion 클래스 사용 예시

In [ ]:
torch.manual_seed(123)  # 랜덤 시드 설정 (재현 가능성 확보)
context_length = batch.shape[1]  # 입력 시퀀스의 길이 (여기서는 6)
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

#### 지금까지 한 일

## 3.6. 싱글 헤드 어텐션을 멀티 헤드 어텐션으로 확장하기

* Multi-head
    - 어텐션 메커니즘을 독립적으로 동작하는 여러 개의 헤드로 나누었다는 의미

### 3.6.1. 여러 개의 싱글 헤드 어텐션 층 쌓기
* 각자 고유한 가중치를 가진 셀프 어텐션 메커니즘을 여러 개 만들고 출력 합치기
* 계산량이 늘어나긴하지만 복잡한 패턴 인식에는 필수적임

* 핵심 아이디어
    - 서로 다른 학습 가능한 선형 투영으로 attetion mechanism을 병렬로 여러 번 실행
    * 선형 투영
        * attention mechanism의 쿼리, 키, 값 벡터와 같은 입력 데이터와 가중치 행렬을 곱한 결과



#### 간단한 MultiHeadAttentionWrapper Class

In [ ]:
class MultiHeadAttentionWrapper(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = torch.nn.ModuleList(
            [
                CausalAttention(
                    d_in, d_out, context_length, dropout, qkv_bias
                )
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

#### 예시
* 2개의 attention head를 사용 (num_heads=2)
* CausalAttention 출력 차원이 2 (d_out = 2)
* MultiHeadAttentionWrapper 클래스는 다음와 같이 4차원 문맥 벡터 출력

#### MultiHeadAttentionWrapper 클래스 사용법

In [ ]:
torch.manual_seed(123)  # 랜덤 시드 설정 (재현 가능성 확보)
context_length = batch.shape[1]  # 토큰 개수
d_in, d_out = 3, 2

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)

print("context_vecs:", context_vecs)
print("context_vecs.shape:", context_vecs.shape)

#### context_vecs.shape: torch.Size([2, 6, 4])
* 첫 번째 차원 = 2
    - 2개의 입력 텍스트를 사용했기 때문에 context_vecs 텐서의 첫 번째 차원은 2
* 두 번째 차원 = 6
    - 각 입력에 6개의 토큰이 있다는 것을 나타냄
* 세 번째 차원 = 4
    - 각 토큰이 4차원 임베딩 벡터인 걸 나타냄

#### 지금까지 만든 MultiHeadAttentionWrapper 클래스 구현
* 정방향 계산에서 [head(x) for head in self.heads]와 같이 순차적으로 계산함
* 헤드를 병렬로 처리할 수 있도록 이 구현을 개선할 수 있음
    - 행렬 곱셈을 통해 모든 어텐션 헤드의 출력을 동시에 계산

### 3.6.2. 가중치 분할로 멀티 헤드 어텐션 구현

* MultiHeadAttentionWrapper와 CausalAttention 클래스를 별도로 관리하는 대신에 MultiHeadAttention 클래스로 결합 가능
    - 더 효율적으로도 구현 가능함 

* MultiHeadAttentionWrapper에서는 개별 어텐션 헤드를 나타내는 CausalAttention 객체가 담긴 리스트(self.heads)를 만들어 여러 개의 헤드 구현

* MultiHeadAttention 클래스는 하나의 클래스 안에 멀티 헤드 기능을 통합
    - 선형 투영된 쿼리, 키, 값 텐서의 크기를 변경하는 방법을 사용하여 입력을 여러 개의 헤드로 나눔
    - 그 후 어텐션을 계산하여 각 헤드의 결과 결합

#### 효율적인 MultiHeadAttention Class


In [ ]:
import torch
import torch.nn as nn

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어 떨어져야 합니다"

        self.d_out = d_out
        self.num_heads = num_heads
        # 원하는 출력 차원에 맞도록 투영 차원을 낮춤
        self.head_dim = d_out // num_heads

        # Query, Key, Value 선형 변환
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 여러 head의 출력을 다시 섞어주는 출력 projection
        # Linear층을 사용해 헤드의 출력을 결합
        self.out_proj = nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)

        # causal mask
        # 위쪽 삼각행렬을 1로 만들어서 미래 토큰을 보지 못하게 함
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        # 텐서 크기: (b, num_tokens, d_out)
        # x: (batch_size, num_tokens, d_in)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # num_heads 차원을 추가함으로써 암묵적으로 행렬 분할
        # 그런 다음 마지막 차원을 num_heads에 맞춰 채움
        # keys, queries, values:
        # (b, num_tokens, d_out)
        # -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)


        # head 차원을 앞으로 이동
        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # attention score 계산
        # queries: (b, num_heads, num_tokens, head_dim)
        # keys.transpose(2, 3): (b, num_heads, head_dim, num_tokens)
        # 결과: (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)

        # 현재 토큰 수에 맞게 mask 자르기
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # 미래 토큰 위치를 -inf로 채움
        # 마스크를 사용해 어텐션 점수를 채움
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # scaled dot-product attention
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5,
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        # attention weight와 value 곱하기
        # attn_weights: (b, num_heads, num_tokens, num_tokens)
        # values:       (b, num_heads, num_tokens, head_dim)
        # 결과:          (b, num_heads, num_tokens, head_dim)
        context_vec = attn_weights @ values

        # 다시 head 차원을 뒤로 보냄
        # 텐서 크기: (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        context_vec = context_vec.transpose(1, 2)

        # 여러 head를 하나로 결합
        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_tokens, d_out)
        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )

        # 마지막 출력 projection (선형 투영)
        context_vec = self.out_proj(context_vec)

        return context_vec

#### 멀티 헤드 층으로 시작한 다음 이 층을 내부에서 개별 어텐션 헤드로 나눔

* d_out 차원을 num_heads와 head_dim으로 나누는 것이 핵심
    - head_dim = d_out / num_heads
    - 이를 통해 (b, num_tokens, d_out) 차원의 텐서를 (b, num_tokens, num_heads, head_dim) 차원으로 변환함
    - 그 후 전치하여 (b, num_heads, num_tokens, head_dim) 크기를 만듦

#### 배치 행렬 곱셈 이해하기

* 이 텐서와 마지막 두 차원 num_tokens와 head_dim을 전치한 텐서 사이에서 배치 행렬 곱셈 수행
* print (a @ a.transpose(2,3))


* MultiHeadAttention에서 어텐션 가중치와 문맥 벡터를 계산한 후 모든 헤드의 문맥 벡터를 전처리하여 (b, num_tokens, num_heads, head_dim) 크기로 되돌림
    - 그 후 벡터 크기를 변경하여 (b, num_tokens, d_out) 크기로 만듦
    - 결과적으로 모든 헤드의 출력을 결합한 효과
* MultiHeadAttention 클래스가 MultiHeadAttentionWrapper보다 훨씬 복잡해 보이지만 |더 효율적인 이유
    - keys = self.W_key(x)와 같이 한 번의 행렬 곱셈으로 키를 계산하기 때문임
    - MultiHeadAttentionWrapper에서는 어텐션 헤드마다 이 행렬 곱셈을 반복

#### MultiHeadAttention 클래스 사용법

In [ ]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print("context_vecs\n:", context_vecs)
print("context_vecs.shape\n:", context_vecs.shape)

#### GPT-2
* 가장 작은 GPT-2 모델(117M)은 어텐션 헤드가 12개, 문맥 벡터의 임베딩 크기 768
* 가장 큰 GPT-2 모델(1.5B)은 어텐션 헤드가 25개, 문맥 벡터의 임베딩 크기 1,600
* GPT 모델에서는 토큰 입력의 임베딩 크기와 문맥의 임베딩 크기가 같음 (d_in = d_out)